In [ ]:
import torch
import numpy as np

from dinosaw.helpers import add_custom_font, get_model, get_models, ModelTypes, model_names #, get_features
from dinosaw.linear_probe import do_linear_probe, RampTypes, LinearProbeResult, get_ramp, gen_sample_mask
from dinosaw.models.vit_wrapper import PretrainedViTWrapper, MODEL_LIST
import dinosaw.utils as utils
from skimage.transform import resize

from os import listdir
from PIL import Image

import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec

from typing import Literal, TypeAlias

SEED = 100001
torch.manual_seed(SEED)
np.random.seed(SEED)
DEVICE = 'cuda:0'

In [ ]:
# selected_model = 'alibi_dv2_coco'
# selected_models: tuple[ModelTypes, ...] = ('dv2', 'dv3', 'alibi_dv2_coco')
selected_models: tuple[ModelTypes, ...] = ('dv_b', 'dv2_b', 'dv3_b', 'clip_b', 'vit_b_in', 'vit_b')

models = get_models(selected_models, '../../trained_models', device=DEVICE,  conf_path='../../dinov3')
S = models[selected_models[0]].stride

n_layers = 12
n_dims = 768

In [ ]:
def get_feature_list(
    model: PretrainedViTWrapper,
    pil_img: Image.Image,
    channel_last: bool = False,
    channel_blank: bool = False,
    to_half: bool = False,
    device: str = "cuda:0",
    S: int = 14,
    N: int = 11
) -> list[np.ndarray]:
    tr = utils.closest_resize(pil_img.height, pil_img.width, model.stride)
    img_tensor = utils.convert_image(pil_img, tr, device_str=device, to_half=to_half)

    is_dv3 = 'dinov3' in model.model_identifier
    with torch.no_grad():
        embs = model.get_intermediate_layers(img_tensor, n=N, Dv3=is_dv3)
    embs_np = [utils.to_numpy(emb.squeeze(0)) for emb in embs]
    if channel_blank:
        channels_to_blank = [47, 113, 117, 359]
        for emb_np in embs_np:
            emb_np[channels_to_blank, :, :] = 0
    if channel_last:
        embs_np = [np.transpose(emb_np, (1, 2, 0)) for emb_np in embs_np]

    return embs_np

In [ ]:
ds_folder = 'data/linear_probe/homog_micros'
image_files = [f for f in listdir(ds_folder)]
n_imgs = len(image_files)

features: dict[ModelTypes, list[list[np.ndarray]]] = {key: [] for key in selected_models}

for key, model in models.items():
    for img_file in image_files:
        img_path = f'{ds_folder}/{img_file}'
        img = Image.open(img_path).convert('RGB')
        feat_list = get_feature_list(model, img, device=DEVICE, channel_last=True, S=S, N=n_layers)
        features[key].append(feat_list)

In [ ]:
AverageResult: TypeAlias = tuple[np.ndarray, np.ndarray, float, float, np.ndarray]
def average_results(results: list[LinearProbeResult]) -> AverageResult:
    channel_scores_arr = np.array([res['per_channel_scores'] for res in results])
    mean_channel_scores = np.mean(channel_scores_arr, axis=0)
    std_channel_scores = np.std(channel_scores_arr, axis=0)

    scores_arr = np.array([res['stack_r_squared'] for res in results])
    mean_score = np.mean(scores_arr, axis=0)
    std_score = np.std(scores_arr, axis=0)

    n_pred_dims = results[0]['stack_pred'].shape[-1]
    mean_pred = np.zeros((34, 34, n_pred_dims))

    for res in results:
        pred = resize(res['stack_pred'], mean_pred.shape, order=1)
        mean_pred += pred / len(results)

    return mean_channel_scores, std_channel_scores, mean_score, std_score, mean_pred

In [ ]:
# ramps: tuple[RampTypes, ...] = ('lr', 'ud', 'diag', 'radial')
ramp: RampTypes = 'lr'
model_to_layer_results: dict[ModelTypes, list[AverageResult]] = {key: [] for key in selected_models}
MASK_CUTOFF_FRAC = 1
STEP = 6
RANDOM_MASK = True

for key, model in models.items():
    for layer in range(n_layers):
        image_results_for_layer = []
        for i in range(n_imgs):
            feats = features[key][i][layer]
            result = do_linear_probe(feats, ramp, probe_by_channel=True, mask_step=STEP, mask_cutoff_frac=MASK_CUTOFF_FRAC, random_mask=RANDOM_MASK)
            image_results_for_layer.append(result)
        averaged = average_results(image_results_for_layer)
        model_to_layer_results[key].append(averaged)
        

In [ ]:
model_to_layer_channel_scores: dict[ModelTypes, np.ndarray] = {key: None for key in selected_models}
for key in selected_models:
    layer_channel_scores = np.array([res[0] for res in model_to_layer_results[key]])
    model_to_layer_channel_scores[key] = layer_channel_scores

In [ ]:
from matplotlib.patches import Rectangle
from matplotlib.collections import PatchCollection

def add_red_square_overlay(ax: plt.Axes, mask: np.ndarray, sf_h: float, sf_w: int) -> None:
    patches: list[Rectangle] = []
    y_inds, x_inds  = np.nonzero(mask)
    for x, y in zip(x_inds, y_inds):
        rect = Rectangle((x - sf_w / 2, y - sf_h / 2), sf_w, sf_h)
        patches.append(rect)
    pc = PatchCollection(patches, color='red', facecolor='none')
    ax.add_collection(pc)

In [ ]:
%%capture
add_custom_font('resources/fonts', 'Grotesk')
# fig, axs = plt.subplots(1, len(selected_models), figsize=(1 * len(selected_models), 5), sharex=True, sharey=True)

n_rows, n_cols = 2, len(selected_models) +1
FS = 15
W, H = len(selected_models), 12

colors: dict[ModelTypes, str] = {
    'dv2': '#5762D5',
    'dv3': '#fcba03',
    'alibi_dv2_coco': '#16ce37',
}


w_spacing = [1/3 for _ in range(len(selected_models))] #[1/3, 1/3, 1/3, 1/4]
w_spacing.append(1/4)
h_spacing = [0.10, 0.6, ]
fig = plt.figure(figsize=(sum([W  * w for w in w_spacing ]), sum([H * h for h in h_spacing])))
gs = GridSpec(n_rows, n_cols, figure=fig, width_ratios=w_spacing, height_ratios=h_spacing, hspace=0.2, wspace=0.5)

h, w = 34, 34
ramp_arr = get_ramp(ramp, h, w)

ramp_ax = fig.add_subplot(gs[0, :-1])
ramp_ax.imshow(ramp_arr, aspect='equal', vmin=0, vmax=1, interpolation='nearest', cmap='viridis',)
ramp_ax.set_xticks([])
ramp_ax.set_yticks([])
ramp_ax.set_title('Target ramp', fontsize=FS)

mask = gen_sample_mask((h, w), ramp, STEP, MASK_CUTOFF_FRAC, random_mask=RANDOM_MASK)
add_red_square_overlay(ramp_ax, mask, 1, 1)

# first_ax = fig.add_subplot(gs[1, 0])
fingerprint_axes = []
for i, key in enumerate(selected_models):
    ax = fig.add_subplot(gs[1, i])
    fingerprint_axes.append(ax)
    ax.imshow(model_to_layer_channel_scores[key].T, aspect='auto', vmin=0, vmax=0.5, cmap='viridis',)
    name = model_names[key]
    name = name.replace('(COCO)', '')
    weight = 700 if 'alibi' in key else 500
    ax.set_title(name, fontsize=FS, weight=weight)
    ax.tick_params(labelsize=FS)
    print(i, key)

    if i > 0:
        ax.set_yticks([])
    else:
        ax.set_ylabel('Channel', fontsize=FS)

fingerprint_axes[2].set_xlabel('Layer', fontsize=FS)
cbar_ax = fig.add_subplot(gs[1, -1])
cbar = fig.colorbar(ax.images[0], ax=cbar_ax, fraction=1, pad=0.08,)
cbar.set_label(r'Per-channel $R^2$ scores', fontsize=FS, rotation=270, labelpad=20)
cbar_ax.set_axis_off()
cbar.ax.tick_params(labelsize=FS)

plt.tight_layout()

# plt.savefig('saved/07.png', dpi=300, bbox_inches='tight')
plt.savefig('saved/07_diff_models.png', dpi=300, bbox_inches='tight', pil_kwargs={'optimize': True})

# cbar.ax.yaxis.set_ticks_position('left')
# cbar.ax.yaxis.set_label_position('left')

# global_score_ax = fig.add_subplot(gs[2, 1:])
# for key in selected_models:
#     r2_scores = np.array([res[2] for res in model_to_layer_results[key]])
#     r2_std = np.array([res[3] for res in model_to_layer_results[key]])
#     name = model_names[key]
#     name = name.replace('(COCO)', '')
#     global_score_ax.plot(r2_scores, label=name, color=colors[key],)
#     global_score_ax.fill_between(
#         np.arange(n_layers),
#         r2_scores - r2_std,
#         r2_scores + r2_std,
#         color=colors[key],
#         alpha=0.3,
#     )

# global_score_ax.set_xlabel('Layer', fontsize=FS)
# global_score_ax.set_ylabel(r'$R^2$ scores', fontsize=FS)
# global_score_ax.tick_params(labelsize=FS-4)
# global_score_ax.legend(fontsize=FS-8)

# plt.tight_layout()